In [6]:
import torch
import torch.nn as nn
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm import tqdm

# ------------------------
# 1. Define Model Class
# ------------------------
# We must define the class again to load the saved model state
class StockTransformer(nn.Module):
    def __init__(self, feature_dim, num_companies):
        super().__init__()
        d_model = 128
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.company_emb = nn.Embedding(num_companies, d_model)
        self.norm = nn.LayerNorm(d_model) # Must match the trained model
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead=4, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, 1))
        
    def forward(self, x, c):
        x = self.input_proj(x)
        x = x + self.company_emb(c).unsqueeze(1) 
        x = self.norm(x)
        x = self.encoder(x)
        return self.head(x[:, -1, :]).squeeze(-1)

# ------------------------
# 2. Setup and Load Files
# ------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Load the checkpoint
ckpt = torch.load("./model_data.pt")
print("✅ Loaded 'model_data.pt'")

# Load the raw test windows
test = pickle.load(open("./windows/test_windows.pkl", "rb"))
print("✅ Loaded 'test_windows.pkl'")

# ------------------------
# 3. Initialize Model and Encoders
# ------------------------
companies = ckpt["companies"]

# Get feature_dim from the test data itself
feature_dim = test[0][0].shape[1]

model = StockTransformer(feature_dim, len(companies)).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print("Model loaded and in eval mode.")

# Create the company-to-index mapping
enc = LabelEncoder().fit(companies)
company_to_idx = {t: enc.transform([t])[0] for t in companies}

# ------------------------
# 4. Run Prediction Loop
# ------------------------
y_true, y_pred, tickers = [], [], []

print("Running predictions on test set...")
for X, y, t in tqdm(test):
    # 't' is the ticker string (e.g., 'AAPL') from test_windows.pkl
    
    # Safety check: clean any NaNs (same as in training)
    X_clean = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    # Convert to tensors
    X_tensor = torch.tensor(X_clean, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    c_tensor = torch.tensor([company_to_idx[t]], dtype=torch.long).to(DEVICE)
    
    with torch.no_grad():
        p = model(X_tensor, c_tensor)
        
    val = p.item()
    
    # Only append valid predictions
    if not np.isnan(val) and np.isfinite(val):
        y_true.append(y)
        y_pred.append(val)
        tickers.append(t)

# ------------------------
# 5. Calculate Metrics
# ------------------------
y_true = np.array(y_true)
y_pred = np.array(y_pred)
tickers = np.array(tickers)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print("\n" + "="*30)
print("     Final Model Metrics")
print("="*30)
print(f"✅ RMSE: {rmse:.4f}")
print(f"✅ MAE:  {mae:.4f}")
print(f"✅ R²:   {r2:.4f}")

# ------------------------
# 6. Save Predictions to CSV
# ------------------------
print("\nSaving predictions to CSV...")

abs_error = np.abs(y_true - y_pred)
# Calculate percentage error, handling cases where y_true is 0
percentage_error = np.where(y_true != 0, abs_error / np.abs(y_true) * 100, 0)

df = pd.DataFrame({
    "Company": tickers,
    "Actual": y_true,
    "Predicted": y_pred,
    "Abs_Error": abs_error,
    "Percentage_Error": percentage_error
})

df.to_csv("predictions_test.csv", index=False)
print("✅ Saved 'predictions_test.csv' with row-wise errors.")

Using device: cuda
✅ Loaded 'model_data.pt'
✅ Loaded 'test_windows.pkl'
Model loaded and in eval mode.
Running predictions on test set...


  0%|          | 0/972 [00:00<?, ?it/s]

100%|██████████| 972/972 [00:01<00:00, 864.65it/s]


     Final Model Metrics
✅ RMSE: 16.8627
✅ MAE:  1.0574
✅ R²:   0.4803

Saving predictions to CSV...
✅ Saved 'predictions_test.csv' with row-wise errors.


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm import tqdm
import os

# ------------------------
# 1. Define Model Class
# ------------------------
# We must define the class again to load the saved model state
class StockTransformer(nn.Module):
    def __init__(self, feature_dim, num_companies):
        super().__init__()
        d_model = 128
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.company_emb = nn.Embedding(num_companies, d_model)
        self.norm = nn.LayerNorm(d_model) # Must match the trained model
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead=4, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, 1))
        
    def forward(self, x, c):
        x = self.input_proj(x)
        x = x + self.company_emb(c).unsqueeze(1) 
        x = self.norm(x)
        x = self.encoder(x)
        return self.head(x[:, -1, :]).squeeze(-1)

# ------------------------
# 2. Setup and Load Files
# ------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = "windows"
print(f"Using device: {DEVICE}")

# Load the checkpoint
try:
    ckpt = torch.load("./model_data.pt")
    print("✅ Loaded 'model_data.pt'")
except FileNotFoundError:
    print("❌ ERROR: 'model_data.pt' not found. Did you run the training script?")
    exit()

# Load the raw test windows
try:
    test = pickle.load(open(os.path.join(OUTPUT_DIR, "test_windows.pkl"), "rb"))
    print("✅ Loaded 'test_windows.pkl'")
except FileNotFoundError:
    print(f"❌ ERROR: 'test_windows.pkl' not found in '{OUTPUT_DIR}'.")
    print("Please run the preprocessing script first.")
    exit()

# ------------------------
# 3. Initialize Model and Encoders
# ------------------------
companies = ckpt["companies"]

# Get feature_dim from the test data itself
try:
    feature_dim = test[0][0].shape[1]
except Exception as e:
    print(f"❌ ERROR: Could not get feature_dim from test data. Error: {e}")
    print("Make sure 'test_windows.pkl' is not empty and has the (X, y, t) structure.")
    exit()

model = StockTransformer(feature_dim, len(companies)).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Model loaded with feature_dim={feature_dim} and num_companies={len(companies)}.")
print("Model is in evaluation mode.")

# Create the company-to-index mapping
enc = LabelEncoder().fit(companies)
company_to_idx = {t: enc.transform([t])[0] for t in companies}

# ------------------------
# 4. Run Prediction Loop
# ------------------------
y_true, y_pred, tickers = [], [], []

print("Running predictions on test set...")
for X, y, t in tqdm(test):
    # 't' is the ticker string (e.g., 'AAPL')
    # 'y' is the true, UN-SCALED target value
    
    # Check if this company was known at training time
    if t not in company_to_idx:
        print(f"Warning: Skipping ticker '{t}' not seen during training.")
        continue
        
    # Safety check: clean any NaNs (same as in training)
    X_clean = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    # Convert to tensors
    X_tensor = torch.tensor(X_clean, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    c_tensor = torch.tensor([company_to_idx[t]], dtype=torch.long).to(DEVICE)
    
    with torch.no_grad():
        p = model(X_tensor, c_tensor)
        
    val = p.item()
    
    # Only append valid predictions
    if not np.isnan(val) and np.isfinite(val):
        y_true.append(y)
        y_pred.append(val)
        tickers.append(t)

# ------------------------
# 5. Calculate Metrics (on Un-scaled Data)
# ------------------------
y_true = np.array(y_true)
y_pred = np.array(y_pred)
tickers = np.array(tickers)

if len(y_true) == 0:
    print("❌ ERROR: No valid predictions were made. Cannot calculate metrics.")
    exit()

# Since data was never scaled, these metrics are in actual dollars
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print("\n" + "="*30)
print("   Final Model Metrics (Un-scaled)")
print("="*30)
print(f"✅ RMSE: ${rmse:.4f} (Avg. Dollar Error)")
print(f"✅ MAE:  ${mae:.4f} (Avg. Abs. Dollar Error)")
print(f"✅ R²:   {r2:.4f}")

# ------------------------
# 6. Save Predictions to CSV
# ------------------------
print("\nSaving un-scaled predictions to CSV...")

abs_error = np.abs(y_true - y_pred)
# Calculate percentage error, handling cases where y_true is 0
percentage_error = np.where(y_true != 0, abs_error / np.abs(y_true) * 100, 0)

df = pd.DataFrame({
    "Company": tickers,
    "Actual": y_true,
    "Predicted": y_pred,
    "Abs_Error": abs_error,
    "Percentage_Error": percentage_error
})

# Sort by the worst predictions
df_sorted = df.sort_values(by="Abs_Error", ascending=False)

df_sorted.to_csv("predictions_test.csv", index=False)
print("✅ Saved 'predictions_test.csv' with row-wise errors.")

Using device: cuda
✅ Loaded 'model_data.pt'
✅ Loaded 'test_windows.pkl'
Model loaded with feature_dim=12 and num_companies=10.
Model is in evaluation mode.
Running predictions on test set...


100%|██████████| 1982/1982 [00:03<00:00, 558.73it/s]



   Final Model Metrics (Un-scaled)
✅ RMSE: $96.6496 (Avg. Dollar Error)
✅ MAE:  $83.0432 (Avg. Abs. Dollar Error)
✅ R²:   -0.2301

Saving un-scaled predictions to CSV...
✅ Saved 'predictions_test.csv' with row-wise errors.
